In [ ]:
from src.requirements import *
from src.audio_handler import *
from src.ssl_large import *
from src.asr_model import *
from src.tokenizer import *
import os, sys

In [ ]:
import seaborn as sns
import gc


class SSLDiagnostics:
    """Memory-efficient diagnostics for SSL features."""
    
    def __init__(self, ssl_model, device='cuda'):
        self.ssl_model = ssl_model
        self.device = device
        self.ssl_model.eval()
        self.ssl_model.to(device)
    
    def analyze_features(self, dataloader, max_batches=50, save_dir='diagnostics'):
        """
        Memory-efficient feature analysis.
        
        Args:
            dataloader: ASR DataLoader
            max_batches: Maximum batches to analyze
            save_dir: Directory to save plots
        """
        import os
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"\n{'='*70}")
        print("SSL FEATURE DIAGNOSTICS (Memory-Efficient)")
        print(f"{'='*70}\n")
        
        # Step 1: Compute statistics incrementally
        print("Step 1: Computing statistics (streaming)...")
        stats = self._compute_statistics_streaming(dataloader, max_batches)
        
        # Step 2: Compute dimensionality (sample-based)
        print("\nStep 2: Analyzing dimensionality (sampled)...")
        dim_analysis = self._analyze_dimensionality_sampled(dataloader, max_samples=50000)
        
        # Step 3: Check collapse (sample-based)
        print("\nStep 3: Checking for collapse (sampled)...")
        collapse_metrics = self._check_collapse_sampled(dataloader, max_samples=10000)
        
        # Step 4: Visualizations
        print("\nStep 4: Generating visualizations...")
        self._create_visualizations(stats, dim_analysis, collapse_metrics, save_dir)
        
        # Print summary
        self._print_summary(stats, dim_analysis, collapse_metrics)
        
        # Clear memory
        gc.collect()
        torch.cuda.empty_cache()
        
        return {
            'statistics': stats,
            'dimensionality': dim_analysis,
            'collapse_metrics': collapse_metrics
        }
    
    def _compute_statistics_streaming(self, dataloader, max_batches):
        """Compute statistics using streaming algorithm (Welford's method)."""
        
        n = 0
        mean = None
        M2 = None
        min_vals = None
        max_vals = None
        
        with torch.no_grad():
            for i, batch in enumerate(tqdm(dataloader, desc="Computing stats", total=max_batches)):
                if i >= max_batches:
                    break
                
                # Get waveforms
                if isinstance(batch, (list, tuple)):
                    waveforms = batch[0]
                else:
                    waveforms = batch
                
                waveforms = waveforms.to(self.device)
                
                # Extract features
                z = self.ssl_model.encoder(waveforms).transpose(1, 2)
                features = self.ssl_model.context(z)
                
                # Flatten to (N, D)
                features_flat = features.reshape(-1, features.size(-1)).cpu().numpy()
                
                # Update statistics (Welford's online algorithm)
                if mean is None:
                    # Initialize
                    feat_dim = features_flat.shape[1]
                    mean = np.zeros(feat_dim)
                    M2 = np.zeros(feat_dim)
                    min_vals = np.full(feat_dim, np.inf)
                    max_vals = np.full(feat_dim, -np.inf)
                
                for sample in features_flat:
                    n += 1
                    delta = sample - mean
                    mean += delta / n
                    delta2 = sample - mean
                    M2 += delta * delta2
                    
                    # Update min/max
                    min_vals = np.minimum(min_vals, sample)
                    max_vals = np.maximum(max_vals, sample)
                
                # Clear GPU memory
                del features, features_flat, waveforms
                torch.cuda.empty_cache()
        
        # Compute final statistics
        variance = M2 / n
        std = np.sqrt(variance)
        
        stats = {
            'mean': mean,
            'std': std,
            'min': min_vals,
            'max': max_vals,
            'n_samples': n,
            'mean_abs': np.mean(np.abs(mean)),
            'std_mean': np.mean(std),
            'std_min': np.min(std),
            'std_max': np.max(std),
            'dead_features': np.sum(std < 0.01),
            'dead_features_pct': np.sum(std < 0.01) / len(std) * 100,
        }
        
        print(f"  ✓ Processed {n:,} samples")
        
        return stats
    
    def _analyze_dimensionality_sampled(self, dataloader, max_samples=50000):
        """Analyze dimensionality using random sampling."""
        
        features_list = []
        total_samples = 0
        
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Sampling for SVD"):
                if total_samples >= max_samples:
                    break
                
                # Get waveforms
                if isinstance(batch, (list, tuple)):
                    waveforms = batch[0]
                else:
                    waveforms = batch
                
                waveforms = waveforms.to(self.device)
                
                # Extract features
                z = self.ssl_model.encoder(waveforms).transpose(1, 2)
                features = self.ssl_model.context(z)
                
                # Flatten and sample
                features_flat = features.reshape(-1, features.size(-1))
                
                # Random sample from this batch
                batch_size = features_flat.size(0)
                sample_size = min(1000, batch_size)  # Sample up to 1000 per batch
                
                if batch_size > sample_size:
                    indices = torch.randperm(batch_size)[:sample_size]
                    features_sampled = features_flat[indices]
                else:
                    features_sampled = features_flat
                
                features_list.append(features_sampled.cpu().numpy())
                total_samples += features_sampled.size(0)
                
                # Clear GPU memory
                del features, features_flat, features_sampled, waveforms
                torch.cuda.empty_cache()
        
        # Concatenate samples
        features_sampled = np.concatenate(features_list, axis=0)
        print(f"  Running SVD on {features_sampled.shape[0]:,} samples...")
        
        # Center the data
        features_centered = features_sampled - features_sampled.mean(axis=0)
        
        # Compute SVD
        U, S, Vt = np.linalg.svd(features_centered, full_matrices=False)
        
        # Compute explained variance
        explained_variance = (S ** 2) / np.sum(S ** 2)
        cumulative_variance = np.cumsum(explained_variance)
        
        # Find ranks
        rank_50 = np.searchsorted(cumulative_variance, 0.50) + 1
        rank_90 = np.searchsorted(cumulative_variance, 0.90) + 1
        rank_95 = np.searchsorted(cumulative_variance, 0.95) + 1
        rank_99 = np.searchsorted(cumulative_variance, 0.99) + 1
        
        dim_analysis = {
            'singular_values': S,
            'explained_variance': explained_variance,
            'cumulative_variance': cumulative_variance,
            'rank_50': rank_50,
            'rank_90': rank_90,
            'rank_95': rank_95,
            'rank_99': rank_99,
            'total_dims': features_sampled.shape[1],
            'effective_rank_ratio': rank_95 / features_sampled.shape[1],
        }
        
        # Clear memory
        del features_sampled, features_centered, U, Vt
        gc.collect()
        
        return dim_analysis
    
    def _check_collapse_sampled(self, dataloader, max_samples=10000):
        """Check for collapse using sampled correlation."""
        
        # Collect samples
        features_list = []
        total_samples = 0
        
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Sampling for correlation"):
                if total_samples >= max_samples:
                    break
                
                # Get waveforms
                if isinstance(batch, (list, tuple)):
                    waveforms = batch[0]
                else:
                    waveforms = batch
                
                waveforms = waveforms.to(self.device)
                
                # Extract features
                z = self.ssl_model.encoder(waveforms).transpose(1, 2)
                features = self.ssl_model.context(z)
                
                # Flatten and sample
                features_flat = features.reshape(-1, features.size(-1))
                
                # Random sample
                batch_size = features_flat.size(0)
                sample_size = min(500, batch_size)
                
                if batch_size > sample_size:
                    indices = torch.randperm(batch_size)[:sample_size]
                    features_sampled = features_flat[indices]
                else:
                    features_sampled = features_flat
                
                features_list.append(features_sampled.cpu().numpy())
                total_samples += features_sampled.size(0)
                
                # Clear GPU memory
                del features, features_flat, features_sampled, waveforms
                torch.cuda.empty_cache()
        
        # Concatenate
        features_sampled = np.concatenate(features_list, axis=0)
        print(f"  Computing correlation on {features_sampled.shape[0]:,} samples...")
        
        # Correlation matrix
        corr_matrix = np.corrcoef(features_sampled.T)
        
        # Remove diagonal
        corr_no_diag = corr_matrix[~np.eye(corr_matrix.shape[0], dtype=bool)]
        
        # High correlation pairs
        high_corr_pairs = np.sum(np.abs(corr_no_diag) > 0.9)
        
        collapse_metrics = {
            'avg_correlation': np.mean(np.abs(corr_no_diag)),
            'max_correlation': np.max(np.abs(corr_no_diag)),
            'high_corr_pairs': high_corr_pairs,
            'correlation_matrix': corr_matrix,
        }
        
        # Collapse score
        score = 0
        if collapse_metrics['avg_correlation'] > 0.7:
            score += 1
        if collapse_metrics['high_corr_pairs'] > 100:
            score += 1
        
        collapse_metrics['collapse_score'] = score
        
        # Clear memory
        del features_sampled, corr_matrix
        gc.collect()
        
        return collapse_metrics
    
    def _create_visualizations(self, stats, dim_analysis, collapse_metrics, save_dir):
        """Create diagnostic plots."""
        
        # 1. Singular values
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        plt.plot(dim_analysis['singular_values'][:50], 'o-', markersize=4)
        plt.xlabel('Component')
        plt.ylabel('Singular Value')
        plt.title('Top 50 Singular Values')
        plt.grid(True, alpha=0.3)
        plt.yscale('log')
        
        plt.subplot(1, 2, 2)
        plt.plot(dim_analysis['cumulative_variance'][:100], 'o-', markersize=3)
        plt.axhline(y=0.95, color='r', linestyle='--', label='95%')
        plt.axhline(y=0.90, color='orange', linestyle='--', label='90%')
        plt.xlabel('Components')
        plt.ylabel('Cumulative Variance')
        plt.title('Explained Variance')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{save_dir}/singular_values.png', dpi=150, bbox_inches='tight')
        print(f"  ✓ Saved: {save_dir}/singular_values.png")
        plt.close()
        
        # 2. Feature statistics
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        plt.hist(stats['std'], bins=50, alpha=0.7, edgecolor='black')
        plt.axvline(x=0.01, color='r', linestyle='--', label='Dead threshold')
        plt.xlabel('Standard Deviation')
        plt.ylabel('Count')
        plt.title('Feature Std Distribution')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.plot(stats['std'], alpha=0.7)
        plt.axhline(y=stats['std_mean'], color='r', linestyle='--', 
                   label=f"Mean: {stats['std_mean']:.3f}")
        plt.axhline(y=0.01, color='orange', linestyle='--', label='Dead')
        plt.xlabel('Feature Dimension')
        plt.ylabel('Std Dev')
        plt.title('Variance per Dimension')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{save_dir}/feature_stats.png', dpi=150, bbox_inches='tight')
        print(f"  ✓ Saved: {save_dir}/feature_stats.png")
        plt.close()
        
        # 3. Correlation heatmap (sample 50 dims)
        plt.figure(figsize=(10, 8))
        
        sample_size = min(50, collapse_metrics['correlation_matrix'].shape[0])
        sample_indices = np.linspace(0, collapse_metrics['correlation_matrix'].shape[0]-1, 
                                    sample_size, dtype=int)
        corr_sample = collapse_metrics['correlation_matrix'][np.ix_(sample_indices, sample_indices)]
        
        sns.heatmap(corr_sample, cmap='coolwarm', center=0, vmin=-1, vmax=1, 
                   square=True, cbar_kws={'label': 'Correlation'})
        plt.title(f'Correlation Matrix (sampled {sample_size} dims)')
        plt.tight_layout()
        plt.savefig(f'{save_dir}/correlation_matrix.png', dpi=150, bbox_inches='tight')
        print(f"  ✓ Saved: {save_dir}/correlation_matrix.png")
        plt.close()
    
    def _print_summary(self, stats, dim_analysis, collapse_metrics):
        """Print summary."""
        print(f"\n{'='*70}")
        print("SUMMARY REPORT")
        print(f"{'='*70}\n")
        
        print("1. BASIC STATISTICS:")
        print(f"   Samples analyzed: {stats['n_samples']:,}")
        print(f"   Feature dimension: {dim_analysis['total_dims']}")
        print(f"   Mean (abs): {stats['mean_abs']:.4f}")
        print(f"   Std (mean): {stats['std_mean']:.4f}")
        
        print(f"\n2. DEAD FEATURES:")
        print(f"   Count: {stats['dead_features']}/{dim_analysis['total_dims']}")
        print(f"   Percentage: {stats['dead_features_pct']:.1f}%")
        if stats['dead_features_pct'] > 20:
            print("   ⚠️  WARNING: >20% dead!")
        else:
            print("   ✓ OK")
        
        print(f"\n3. EFFECTIVE DIMENSIONALITY:")
        print(f"   Rank (95%): {dim_analysis['rank_95']}/{dim_analysis['total_dims']}")
        print(f"   Rank (90%): {dim_analysis['rank_90']}/{dim_analysis['total_dims']}")
        print(f"   Ratio: {dim_analysis['effective_rank_ratio']:.2%}")
        
        if dim_analysis['rank_95'] < 20:
            print("   ❌ TOO LOW for ASR!")
        elif dim_analysis['rank_95'] < 40:
            print("   ⚠️  Marginal")
        else:
            print("   ✓ GOOD")
        
        print(f"\n4. CORRELATION:")
        print(f"   Average: {collapse_metrics['avg_correlation']:.4f}")
        print(f"   High pairs: {collapse_metrics['high_corr_pairs']}")
        
        print(f"\n5. COLLAPSE SCORE: {collapse_metrics['collapse_score']}/4")
        if collapse_metrics['collapse_score'] <= 1:
            print("   ✓✓ GOOD")
        elif collapse_metrics['collapse_score'] == 2:
            print("   ⚠️  WARNING")
        else:
            print("   ❌ CRITICAL")
        
        print(f"\n{'='*70}\n")

In [ ]:
class ASRDiagnostics:
    """Memory-efficient ASR diagnostics."""
    
    def __init__(self, asr_model, device='cuda'):
        self.asr_model = asr_model
        self.device = device
        self.asr_model.eval()
        self.asr_model.to(device)
    
    def analyze_asr_features(self, dataloader, max_batches=20):
        """Analyze ASR feature flow (memory-efficient)."""
        
        print(f"\n{'='*70}")
        print("ASR FEATURE FLOW ANALYSIS")
        print(f"{'='*70}\n")
        
        # Compute rank for each stage incrementally
        stages_info = {}
        
        for stage_name in ['SSL', 'Multiscale', 'Diversified', 'LSTM', 'Projected']:
            print(f"Analyzing {stage_name}...")
            rank, mean_std, shape = self._analyze_stage(dataloader, stage_name, max_batches)
            stages_info[stage_name] = {
                'rank': rank,
                'mean_std': mean_std,
                'shape': shape
            }
            
            # Clear memory
            gc.collect()
            torch.cuda.empty_cache()
        
        # Print results
        print(f"\n{'Stage':<15} {'Shape':<20} {'Rank (95%)':<15} {'Mean Std':<15}")
        print("-" * 65)
        
        for name, info in stages_info.items():
            shape_str = str(info['shape'])
            rank_str = f"{info['rank']}/{info['shape'][1]}"
            print(f"{name:<15} {shape_str:<20} {rank_str:<15} {info['mean_std']:<15.4f}")
        
        print(f"\n{'='*70}\n")
        
        return stages_info
    
    def _analyze_stage(self, dataloader, stage_name, max_batches, max_samples=20000):
        """Analyze a single stage."""
        
        features_list = []
        total_samples = 0
        
        with torch.no_grad():
            for i, batch in enumerate(dataloader):
                if i >= max_batches or total_samples >= max_samples:
                    break
                
                waveforms = batch[0].to(self.device)
                input_lengths = batch[2].to(self.device) if len(batch) > 2 else None
                
                # Extract features up to this stage
                z = self.asr_model.encoder(waveforms).transpose(1, 2)
                ssl_feat = self.asr_model.context(z)
                
                if stage_name == 'SSL':
                    features = ssl_feat
                elif stage_name == 'Multiscale':
                    features = self.asr_model.multiscale(ssl_feat)
                elif stage_name == 'Diversified':
                    multiscale = self.asr_model.multiscale(ssl_feat)
                    features = multiscale + self.asr_model.diversify(multiscale)
                elif stage_name == 'LSTM':
                    multiscale = self.asr_model.multiscale(ssl_feat)
                    diversified = multiscale + self.asr_model.diversify(multiscale)
                    if input_lengths is not None:
                        packed = nn.utils.rnn.pack_padded_sequence(
                            diversified, input_lengths.cpu(), batch_first=True, enforce_sorted=False
                        )
                        lstm_out, _ = self.asr_model.lstm(packed)
                        features, _ = nn.utils.rnn.pad_packed_sequence(lstm_out, batch_first=True)
                    else:
                        features, _ = self.asr_model.lstm(diversified)
                elif stage_name == 'Projected':
                    multiscale = self.asr_model.multiscale(ssl_feat)
                    diversified = multiscale + self.asr_model.diversify(multiscale)
                    if input_lengths is not None:
                        packed = nn.utils.rnn.pack_padded_sequence(
                            diversified, input_lengths.cpu(), batch_first=True, enforce_sorted=False
                        )
                        lstm_out, _ = self.asr_model.lstm(packed)
                        lstm_feat, _ = nn.utils.rnn.pad_packed_sequence(lstm_out, batch_first=True)
                    else:
                        lstm_feat, _ = self.asr_model.lstm(diversified)
                    features = self.asr_model.projection(lstm_feat)
                
                # Sample and collect
                feat_flat = features.reshape(-1, features.size(-1))
                sample_size = min(500, feat_flat.size(0))
                
                if feat_flat.size(0) > sample_size:
                    indices = torch.randperm(feat_flat.size(0))[:sample_size]
                    feat_sampled = feat_flat[indices]
                else:
                    feat_sampled = feat_flat
                
                features_list.append(feat_sampled.cpu().numpy())
                total_samples += feat_sampled.size(0)
                
                # Clear GPU
                del features, feat_flat, feat_sampled, waveforms
                torch.cuda.empty_cache()
        
        # Combine samples
        all_features = np.concatenate(features_list, axis=0)
        
        # Compute statistics
        mean_std = np.mean(np.std(all_features, axis=0))
        
        # Compute rank
        features_centered = all_features - all_features.mean(axis=0)
        U, S, Vt = np.linalg.svd(features_centered, full_matrices=False)
        explained_var = (S ** 2) / np.sum(S ** 2)
        cum_var = np.cumsum(explained_var)
        rank_95 = np.searchsorted(cum_var, 0.95) + 1
        
        shape = all_features.shape
        
        # Clear memory
        del all_features, features_centered, U, S, Vt
        gc.collect()
        
        return rank_95, mean_std, shape

In [ ]:
data_path = os.path.join("data", "metadata_normal.tsv")
token_path = os.path.join("data", "tokenizer.json")
cache_path = os.path.join("data", "cache_mmap", "asr")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Load SSL model
print("Loading SSL model...")
ssl_model = LargeSSLModel(feat_dim=256, proj_dim=256)
checkpoint = torch.load('models/ssl_model/ssl_model_checkpoint_100000.pth', map_location=device)
ssl_model.load_state_dict(checkpoint['model_state_dict'])
ssl_model.to(device)
ssl_model.eval()

In [ ]:
tokenizer = Tokenizer.load(token_path)

asr_dataset = asr_dataset = ASRDataset(
    metadata_path=data_path,
    tokenizer=tokenizer,
    cache_dir=cache_path,
    top_db=TOP_DB
)

asr_dl = DataLoader(
    asr_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_padding_asr,
    pin_memory=True
)

In [ ]:
# Clear memory first
gc.collect()
torch.cuda.empty_cache()

# Load models
device = 'cuda'
ssl_model = LargeSSLModel()
checkpoint = torch.load('models/ssl_model/ssl_model_checkpoint_100000.pth', map_location=device)
ssl_model.load_state_dict(checkpoint['model_state_dict'])
ssl_model.to(device)

# Create DataLoader with smaller batch size
asr_dataset = asr_dataset = ASRDataset(
    metadata_path=data_path,
    tokenizer=tokenizer,
    cache_dir=cache_path,
    top_db=TOP_DB
)

asr_dl = DataLoader(
    asr_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_padding_asr,
    pin_memory=True
)

# Run diagnostics
ssl_diag = SSLDiagnostics(ssl_model, device=device)
results = ssl_diag.analyze_features(
    dataloader=asr_dl,
    max_batches=30,
    save_dir='diagnostics/ssl'
)

# Clear memory
del ssl_diag
gc.collect()
torch.cuda.empty_cache()

# Run ASR diagnostics
asr_model = create_asr_model(
    ssl_checkpoint_path='models/ssl_model/ssl_model_final_50000.pth',
    tokenizer=tokenizer,
    device=device
)

asr_diag = ASRDiagnostics(asr_model, device=device)
asr_results = asr_diag.analyze_asr_features(
    dataloader=asr_dl,
    max_batches=15
)

In [ ]:
def vicreg_loss_debug(z1, z2, lambda_inv=25, lambda_var=25, lambda_cov=1):
    """VICReg with detailed debugging."""
    N, D = z1.shape
    
    # Check for collapse first
    z1_std = z1.std(dim=0).mean().item()
    z2_std = z2.std(dim=0).mean().item()
    
    print(f"\n  Feature std: z1={z1_std:.4f}, z2={z2_std:.4f}")
    
    if z1_std < 0.1 or z2_std < 0.1:
        print(f"  ❌ COLLAPSE: Features have very low variance!")
    
    # 1. Invariance
    invariance_loss = F.mse_loss(z1, z2)
    
    # 2. Variance
    std_z1 = torch.sqrt(z1.var(dim=0) + 1e-4)
    std_z2 = torch.sqrt(z2.var(dim=0) + 1e-4)
    variance_loss = torch.mean(F.relu(1 - std_z1)) + torch.mean(F.relu(1 - std_z2))
    
    # 3. Covariance
    z1_centered = z1 - z1.mean(dim=0)
    z2_centered = z2 - z2.mean(dim=0)
    
    cov_z1 = (z1_centered.T @ z1_centered) / (N - 1)
    cov_z2 = (z2_centered.T @ z2_centered) / (N - 1)
    
    off_diag_z1 = cov_z1.flatten()[:-1].view(D - 1, D + 1)[:, 1:].flatten()
    off_diag_z2 = cov_z2.flatten()[:-1].view(D - 1, D + 1)[:, 1:].flatten()
    
    covariance_loss = off_diag_z1.pow(2).sum() / D + off_diag_z2.pow(2).sum() / D
    
    # Total
    total_loss = (lambda_inv * invariance_loss + 
                  lambda_var * variance_loss + 
                  lambda_cov * covariance_loss)
    
    # Print breakdown
    print(f"  Inv: {invariance_loss.item():.4f} (×{lambda_inv} = {lambda_inv*invariance_loss.item():.2f})")
    print(f"  Var: {variance_loss.item():.4f} (×{lambda_var} = {lambda_var*variance_loss.item():.2f})")
    print(f"  Cov: {covariance_loss.item():.4f} (×{lambda_cov} = {lambda_cov*covariance_loss.item():.2f})")
    print(f"  Total: {total_loss.item():.2f}")
    
    # Check for collapse patterns
    if variance_loss.item() > 0.8:
        print(f"  ⚠️  High variance loss - features collapsing to low std!")
    
    if invariance_loss.item() < 0.01:
        print(f"  ⚠️  Very low invariance loss - views too similar!")
    
    return total_loss

In [ ]:
# 1. Check augmentation quality
print("\n" + "="*70)
print("EMERGENCY DIAGNOSTIC")
print("="*70)

model.eval()

# Get a batch
batch = next(iter(train_dl))
x = batch.to(device)

# Check augmentations
print("\n1. AUGMENTATION CHECK:")
with torch.no_grad():
    x1 = model.augment(x, augment_prob=0.8)
    x2 = model.augment(x, augment_prob=0.8)
    
    diff = (x1 - x2).abs().mean().item()
    orig_scale = x.abs().mean().item()
    
    print(f"   Original scale: {orig_scale:.6f}")
    print(f"   Difference: {diff:.6f}")
    print(f"   Ratio: {diff/orig_scale:.4f}")
    
    if diff/orig_scale < 0.01:
        print("   ❌ PROBLEM: Augmentations barely changing the input!")

# Check encoded features
print("\n2. FEATURE CHECK:")
with torch.no_grad():
    z1 = model.encoder(x1).transpose(1, 2)
    z2 = model.encoder(x2).transpose(1, 2)
    
    # Process through context
    c1 = model.context(z1)
    c2 = model.context_m(z2)
    
    # Project
    p1 = model.projector(c1)
    h2 = model.projector_m(c2)
    
    # Check statistics
    print(f"   p1 mean: {p1.mean().item():.4f}, std: {p1.std().item():.4f}")
    print(f"   h2 mean: {h2.mean().item():.4f}, std: {h2.std().item():.4f}")
    
    # Check similarity
    p1_flat = p1.reshape(-1, p1.size(-1))
    h2_flat = h2.reshape(-1, h2.size(-1))
    
    p1_norm = F.normalize(p1_flat, dim=-1)
    h2_norm = F.normalize(h2_flat, dim=-1)
    
    similarity = (p1_norm * h2_norm).sum(dim=-1).mean().item()
    print(f"   Cosine similarity: {similarity:.4f}")
    
    if similarity > 0.95:
        print("   ❌ PROBLEM: Features are nearly identical!")
    
    # Check for constant features
    dead_dims_p1 = (p1_flat.std(dim=0) < 0.01).sum().item()
    dead_dims_h2 = (h2_flat.std(dim=0) < 0.01).sum().item()
    
    print(f"   Dead dimensions p1: {dead_dims_p1}/{p1_flat.size(1)}")
    print(f"   Dead dimensions h2: {dead_dims_h2}/{h2_flat.size(1)}")
    
    if dead_dims_p1 > 50 or dead_dims_h2 > 50:
        print("   ❌ PROBLEM: Many features collapsed!")

# Check loss components
print("\n3. LOSS COMPONENT CHECK:")
model.train()
with torch.no_grad():
    # Get features
    z1 = model.encoder(x1).transpose(1, 2)
    mask1 = compute_mask_indices(z1.size(0), z1.size(1), 0.065, 10, z1.device)
    z1_masked = apply_mask(z1, mask1)
    
    c1 = model.context(z1_masked)
    p1 = model.predictor(model.projector(c1))
    
    z2 = model.encoder_m(x2).transpose(1, 2)
    c2 = model.context_m(z2)
    h2 = model.projector_m(c2)
    
    p1_masked = p1[mask1]
    h2_masked = h2[mask1]
    
    # Check with debug loss
    loss = vicreg_loss_debug(p1_masked, h2_masked)

print("\n" + "="*70)

In [ ]:
import re
from itertools import groupby


def decode_predictions(log_probs, tokenizer, blank_id=0):
    """
    CTC greedy decoder.
    
    Args:
        log_probs: (seq_len, batch, vocab) - model output
        tokenizer: Your tokenizer
        blank_id: CTC blank token id
    
    Returns:
        List of decoded strings
    """
    # Get most likely token at each step
    # (seq_len, batch)
    predictions = log_probs.argmax(dim=-1).transpose(0, 1)
    
    decoded = []
    for pred in predictions:
        # CTC decode: remove blanks and consecutive duplicates
        pred = pred.tolist()
        
        # Remove consecutive duplicates
        pred = [k for k, _ in groupby(pred)]
        
        # Remove blanks
        pred = [p for p in pred if p != blank_id]
        
        # Convert to string
        text = tokenizer.decode(pred)
        decoded.append(text)
    
    return decoded

In [ ]:
def compute_wer(reference, hypothesis):
    """
    Compute Word Error Rate.
    
    Args:
        reference: Ground truth string
        hypothesis: Predicted string
    
    Returns:
        WER as float (0.0 = perfect, 1.0 = all wrong)
    """
    ref_words = reference.strip().split()
    hyp_words = hypothesis.strip().split()
    
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    
    # Dynamic programming
    d = np.zeros((len(ref_words) + 1, len(hyp_words) + 1))
    
    for i in range(len(ref_words) + 1):
        d[i][0] = i
    for j in range(len(hyp_words) + 1):
        d[0][j] = j
    
    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            if ref_words[i-1] == hyp_words[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                d[i][j] = min(
                    d[i-1][j] + 1,    # deletion
                    d[i][j-1] + 1,    # insertion
                    d[i-1][j-1] + 1   # substitution
                )
    
    return d[len(ref_words)][len(hyp_words)] / len(ref_words)

In [ ]:
def compute_cer(reference, hypothesis):
    """
    Compute Character Error Rate.
    
    Args:
        reference: Ground truth string
        hypothesis: Predicted string
    
    Returns:
        CER as float
    """
    ref_chars = list(reference.replace(' ', ''))
    hyp_chars = list(hypothesis.replace(' ', ''))
    
    if len(ref_chars) == 0:
        return 0.0 if len(hyp_chars) == 0 else 1.0
    
    # Dynamic programming
    d = np.zeros((len(ref_chars) + 1, len(hyp_chars) + 1))
    
    for i in range(len(ref_chars) + 1):
        d[i][0] = i
    for j in range(len(hyp_chars) + 1):
        d[0][j] = j
    
    for i in range(1, len(ref_chars) + 1):
        for j in range(1, len(hyp_chars) + 1):
            if ref_chars[i-1] == hyp_chars[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                d[i][j] = min(
                    d[i-1][j] + 1,
                    d[i][j-1] + 1,
                    d[i-1][j-1] + 1
                )
    
    return d[len(ref_chars)][len(hyp_chars)] / len(ref_chars)

In [ ]:
def evaluate_asr(model, dataloader, tokenizer, device, num_samples=10):
    """
    Evaluate ASR model with WER/CER metrics.
    
    Args:
        model: ASR model
        dataloader: Validation dataloader
        tokenizer: Character tokenizer
        device: Device
        num_samples: Number of samples to print
    
    Returns:
        Dict with avg_loss, wer, cer
    """
    model.eval()
    
    total_loss = 0
    total_wer = 0
    total_cer = 0
    num_batches = 0
    num_samples_evaluated = 0
    
    ctc_loss = torch.nn.CTCLoss(blank=0, zero_infinity=True)
    
    printed_samples = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            waveforms, targets, input_lengths, target_lengths = batch
            
            waveforms = waveforms.to(device)
            input_lengths = input_lengths.to(device)
            
            # Forward pass
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                log_probs = model(waveforms, input_lengths)
            
            # CTC loss
            targets_cat = torch.cat([t for t in targets])
            loss = ctc_loss(
                log_probs,
                targets_cat,
                input_lengths,
                target_lengths
            )
            total_loss += loss.item()
            num_batches += 1
            
            # Decode predictions
            predictions = decode_predictions(log_probs, tokenizer)
            
            # Compute WER/CER for each sample
            for pred, target, target_len in zip(predictions, targets, target_lengths):
                # Decode target
                ref = tokenizer.decode(target.tolist()[:target_len])
                
                wer = compute_wer(ref, pred)
                cer = compute_cer(ref, pred)
                
                total_wer += wer
                total_cer += cer
                num_samples_evaluated += 1
                
                # Print sample predictions
                if printed_samples < num_samples:
                    print(f"\n  REF: {repr(ref)}")
                    print(f"  HYP: {repr(pred)}")
                    print(f"  WER: {wer:.2%}  CER: {cer:.2%}")
                    printed_samples += 1
    
    avg_loss = total_loss / num_batches
    avg_wer = total_wer / num_samples_evaluated
    avg_cer = total_cer / num_samples_evaluated
    
    print(f"\n{'='*60}")
    print(f"EVALUATION RESULTS")
    print(f"{'='*60}")
    print(f"  Loss: {avg_loss:.4f}")
    print(f"  WER:  {avg_wer:.2%}")
    print(f"  CER:  {avg_cer:.2%}")
    print(f"  Samples evaluated: {num_samples_evaluated}")
    
    # Interpret results
    print(f"\n  WER Assessment:")
    if avg_wer < 0.10:
        print(f"  ✓✓✓ Excellent (<10%)")
    elif avg_wer < 0.20:
        print(f"  ✓✓ Good (10-20%)")
    elif avg_wer < 0.30:
        print(f"  ✓ Acceptable (20-30%)")
    elif avg_wer < 0.50:
        print(f"  ⚠️  Poor (30-50%)")
    else:
        print(f"  ❌ Very poor (>50%)")
    
    print(f"\n  CER Assessment:")
    if avg_cer < 0.05:
        print(f"  ✓✓✓ Excellent (<5%)")
    elif avg_cer < 0.10:
        print(f"  ✓✓ Good (5-10%)")
    elif avg_cer < 0.20:
        print(f"  ✓ Acceptable (10-20%)")
    elif avg_cer < 0.40:
        print(f"  ⚠️  Poor (20-40%)")
    else:
        print(f"  ❌ Very poor (>40%)")
    
    print(f"{'='*60}\n")
    
    return {
        'loss': avg_loss,
        'wer': avg_wer,
        'cer': avg_cer,
        'samples': num_samples_evaluated
    }

In [ ]:
# Run evaluation NOW on your current model
print("Evaluating current ASR model...")
results = evaluate_asr(
    model=asr_model,
    dataloader=asr_dl,
    tokenizer=tokenizer,
    device=device,
    num_samples=10  # Print 10 example predictions
)

In [ ]:
def train_asr_with_metrics(asr_model, train_dl, val_dl, optimizer, 
                            scheduler, tokenizer, device,
                            epochs=20, start_epoch=0):
    """
    ASR training with proper WER/CER evaluation.
    """
    ctc_loss_fn = torch.nn.CTCLoss(blank=0, zero_infinity=True)
    scaler = GradScaler()
    
    best_wer = float('inf')
    best_cer = float('inf')
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'wer': [],
        'cer': []
    }
    
    for epoch in range(start_epoch, epochs):
        # Training
        asr_model.train()
        if hasattr(asr_model, 'encoder'):
            asr_model.encoder.eval()  # Keep SSL frozen
        if hasattr(asr_model, 'context'):
            asr_model.context.eval()
        
        total_loss = 0
        num_batches = 0
        
        for i, batch in enumerate(tqdm(train_dl, desc=f"Epoch {epoch+1}/{epochs}")):
            waveforms, targets, input_lengths, target_lengths = batch
            
            waveforms = waveforms.to(device, non_blocking=True)
            input_lengths = input_lengths.to(device)
            
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                log_probs = asr_model(waveforms, input_lengths)
                
                targets_cat = torch.cat([t for t in targets])
                loss = ctc_loss_fn(
                    log_probs,
                    targets_cat,
                    input_lengths,
                    target_lengths
                )
            
            scaler.scale(loss).backward()
            
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                asr_model.parameters(), max_norm=1.0
            )
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_train_loss = total_loss / num_batches
        history['train_loss'].append(avg_train_loss)
        
        # Evaluate every epoch
        print(f"\nEpoch {epoch+1} - Train Loss: {avg_train_loss:.4f}")
        
        results = evaluate_asr(
            asr_model, val_dl, tokenizer, device, num_samples=5
        )
        
        history['val_loss'].append(results['loss'])
        history['wer'].append(results['wer'])
        history['cer'].append(results['cer'])
        
        # Save best model
        if results['wer'] < best_wer:
            best_wer = results['wer']
            best_cer = results['cer']
            
            save_asr_checkpoint(
                asr_model, optimizer, scheduler, epoch,
                'models/asr_model/asr_best.pth',
                best_wer=best_wer,
                best_cer=best_cer
            )
            print(f"✓ New best model! WER: {best_wer:.2%}, CER: {best_cer:.2%}")
        
        torch.cuda.empty_cache()
    
    # Plot training history
    plot_training_history(history)
    
    print(f"\n{'='*60}")
    print(f"TRAINING COMPLETE")
    print(f"  Best WER: {best_wer:.2%}")
    print(f"  Best CER: {best_cer:.2%}")
    print(f"{'='*60}")
    
    return asr_model, history

In [ ]:
def plot_training_history(history):
    """Plot training curves."""
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('CTC Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # WER
    axes[1].plot(history['wer'], 'r-o', markersize=4)
    axes[1].set_title('Word Error Rate (WER)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('WER')
    axes[1].yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{x:.0%}')
    )
    axes[1].grid(True, alpha=0.3)
    
    # CER
    axes[2].plot(history['cer'], 'g-o', markersize=4)
    axes[2].set_title('Character Error Rate (CER)')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('CER')
    axes[2].yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{x:.0%}')
    )
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('diagnostics/asr_training_history.png', dpi=150)
    plt.close()
    print("✓ Saved: diagnostics/asr_training_history.png")